# D+7 활동 예측용 초기 24시간 힌트 피처 생성

## 분석 목적

공통 Base Mart의 사용자를 기준으로 **최초 PING 발신 시점(T0)부터 24시간 동안 확인 가능한 힌트·수신 관련 피처**를 생성한다.

- 최종 타깃: `retained_d7` — T0+7일 이상 T0+8일 미만 PING 발신 여부
- 분석 단위: 사용자 1명당 1행
- 피처 관찰 구간: `[feature_start_at, feature_end_at)` = `[T0, T0+24시간)`
- 시간대: KST (`Asia/Seoul`)
- 결과 테이블: `your-gcp-project.mart.fct_ml_hint_24h`

> 이 노트북은 모델을 학습하지 않는다. 공통 모집단과 시간창을 유지하면서 힌트 피처 테이블을 만드는 것이 목적이다.

## 1. 환경 설정

Colab에서 Google 계정을 인증하고 BigQuery 클라이언트를 생성한다.

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd

PROJECT_ID = "your-gcp-project"
client = bigquery.Client(project=PROJECT_ID)

pd.set_option("display.max_columns", None)
print("BigQuery 연결 완료")

## 2. 원천 데이터 확인

`mart_activation`에서 사용할 수 있는 핵심 컬럼과 자료형을 확인한다.

- `user_id`: 사용자 식별자
- `first_receive_at`: 서비스 이용 중 최초 PING 수신 시각
- `first_hint_opened_at`: 서비스 이용 중 최초 힌트 열람 시각

두 시각 컬럼은 `DATETIME`이므로 이후 `TIMESTAMP(..., 'Asia/Seoul')`로 변환하여 Base Mart의 `TIMESTAMP`와 비교한다.

In [ ]:
query = """
SELECT
    column_name,
    data_type
FROM `your-gcp-project.mart.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'mart_activation'
  AND column_name IN (
      'user_id',
      'first_receive_at',
      'first_hint_opened_at',
      'has_hint_opened',
      'time_to_hint_open_hours'
  )
ORDER BY column_name
"""

activation_schema = client.query(query).to_dataframe()
activation_schema

### 힌트 이벤트 후보 탐색

첫 24시간의 **힌트 열람 횟수**를 만들려면 열람 건별 발생 시각이 필요하다. 관련 컬럼과 이벤트 로그 소스 이벤트를 탐색하여 이벤트 단위 로그가 존재하는지 확인한다.

판단 기준은 다음과 같다.

- `opened_times`, `has_hint_opened` 같은 현재 누적값·스냅샷만으로는 T0 이후 24시간 내 발생한 행동인지 구분할 수 없다.
- 의미가 불명확한 이벤트는 이름만 보고 힌트 열람으로 간주하지 않는다.
- Base Mart의 피처 관찰 기간을 덮지 못하는 로그는 최종 피처에 사용하지 않는다.

In [ ]:
query = """
SELECT 'votes' AS dataset_name, table_name, column_name, data_type
FROM `your-gcp-project.raw.INFORMATION_SCHEMA.COLUMNS`
WHERE REGEXP_CONTAINS(LOWER(column_name), r'hint|open')

UNION ALL

SELECT 'stage' AS dataset_name, table_name, column_name, data_type
FROM `your-gcp-project.stage.INFORMATION_SCHEMA.COLUMNS`
WHERE REGEXP_CONTAINS(LOWER(column_name), r'hint|open')

UNION ALL

SELECT 'mart' AS dataset_name, table_name, column_name, data_type
FROM `your-gcp-project.mart.INFORMATION_SCHEMA.COLUMNS`
WHERE REGEXP_CONTAINS(LOWER(column_name), r'hint|open')

UNION ALL

SELECT 'events' AS dataset_name, table_name, column_name, data_type
FROM `your-gcp-project.events.INFORMATION_SCHEMA.COLUMNS`
WHERE REGEXP_CONTAINS(LOWER(column_name), r'hint|open')

ORDER BY dataset_name, table_name, column_name
"""

hint_columns = client.query(query).to_dataframe()
hint_columns

In [ ]:
query = """
SELECT
    event_key,
    COUNT(*) AS event_count,
    MIN(event_datetime) AS min_event_at,
    MAX(event_datetime) AS max_event_at
FROM `your-gcp-project.events.event_log`
WHERE REGEXP_CONTAINS(LOWER(event_key), r'hint|open')
GROUP BY event_key
ORDER BY event_count DESC
"""

hint_event_keys = client.query(query).to_dataframe()
hint_event_keys

### 데이터 탐색 결론

현재 확인된 데이터에는 힌트 열람 건별 시각이 없다. 따라서 다음 항목은 정확하게 만들 수 없다.

- `hint_open_count_24h`: 첫 24시간 힌트 열람 횟수
- `used_hint_24h`: 첫 24시간 중 어느 시점이든 힌트를 열었는지 여부

대신 `first_hint_opened_at`을 이용하여 **서비스 최초 힌트 열람이 T0 이후 24시간 안에 발생했는지**를 만든다. 비슷해 보이지만 의미가 다르므로 컬럼명을 `first_hint_opened_in_24h`로 명확히 구분한다.

이벤트 로그 소스의 `click_question_open`은 힌트 열람을 의미한다고 확정할 수 없고 Base Mart의 전체 피처 기간도 덮지 못하므로 사용하지 않는다.

## 3. 조인 전 데이터 품질 확인

`mart_activation`이 사용자당 1행인지 확인한다. 중복이 있으면 Base Mart와 조인할 때 행이 늘어나므로, `duplicate_count = 0`이어야 한다.

In [ ]:
query = """
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT user_id) AS user_count,
    COUNT(*) - COUNT(DISTINCT user_id) AS duplicate_count
FROM `your-gcp-project.mart.mart_activation`
"""

activation_qa = client.query(query).to_dataframe()
activation_qa

## 4. Base Mart 기준 분포 확인

Base Mart를 왼쪽 테이블로 두고 `LEFT JOIN`한다. 이를 통해 최종 모집단 4,844명을 유지하면서 다음을 확인한다.

- `mart_activation`과 매칭되지 않는 사용자
- 최초 수신·힌트 열람이 T0 이전, 첫 24시간, 24시간 이후 중 어디에 속하는지
- 최초 수신·힌트 열람 시각이 없는 사용자

주의: `never_received`, `never_hint_opened`에는 `mart_activation` 미매칭 사용자도 포함된다. 실제 행동 이력 없음과 데이터 미매칭을 구분하기 위해 최종 테이블에 `activation_match_flag`를 QA 컬럼으로 보관한다.

In [ ]:
query = """
WITH activation_ts AS (
    SELECT
        user_id,
        TIMESTAMP(first_receive_at, 'Asia/Seoul') AS first_receive_at_ts,
        TIMESTAMP(first_hint_opened_at, 'Asia/Seoul') AS first_hint_opened_at_ts
    FROM `your-gcp-project.mart.mart_activation`
)

SELECT
    COUNT(*) AS base_user_count,
    COUNTIF(a.user_id IS NULL) AS no_activation_match,

    COUNTIF(a.first_receive_at_ts < b.feature_start_at)
        AS received_before_t0,
    COUNTIF(
        a.first_receive_at_ts >= b.feature_start_at
        AND a.first_receive_at_ts < b.feature_end_at
    ) AS first_receive_in_24h,
    COUNTIF(a.first_receive_at_ts >= b.feature_end_at)
        AS first_receive_after_24h,
    COUNTIF(a.user_id IS NOT NULL AND a.first_receive_at_ts IS NULL)
        AS matched_but_never_received,

    COUNTIF(a.first_hint_opened_at_ts < b.feature_start_at)
        AS hint_before_t0,
    COUNTIF(
        a.first_hint_opened_at_ts >= b.feature_start_at
        AND a.first_hint_opened_at_ts < b.feature_end_at
    ) AS first_hint_in_24h,
    COUNTIF(a.first_hint_opened_at_ts >= b.feature_end_at)
        AS first_hint_after_24h,
    COUNTIF(a.user_id IS NOT NULL AND a.first_hint_opened_at_ts IS NULL)
        AS matched_but_never_hint_opened

FROM `your-gcp-project.mart.mart_ml_d7_base` b
LEFT JOIN activation_ts a USING (user_id)
"""

hint_check_df = client.query(query).to_dataframe()
hint_check_df.T

## 5. 최종 피처 정의

| 컬럼 | 정의 | 모델 사용 |
|---|---|---|
| `activation_match_flag` | `mart_activation` 매칭 여부 | 제외, QA용 |
| `had_received_before_t0` | T0 이전 최초 PING 수신 기록 여부 | 사용 |
| `first_receive_in_24h` | 서비스 최초 PING 수신이 `[T0, T0+24h)`에 발생했는지 | 사용 |
| `time_to_first_receive_hours` | T0부터 24시간 내 최초 수신까지 걸린 시간 | 사용 |
| `had_hint_before_t0` | T0 이전 최초 힌트 열람 기록 여부 | 사용 |
| `first_hint_opened_in_24h` | 서비스 최초 힌트 열람이 `[T0, T0+24h)`에 발생했는지 | 사용 |
| `time_to_first_hint_hours` | T0부터 24시간 내 최초 힌트 열람까지 걸린 시간 | 사용 |
| `received_before_first_hint` | 24시간 내 최초 힌트 열람 전에 PING 수신 경험이 있었는지 | 사용 |

### 결측 처리 원칙

`time_to_first_receive_hours`와 `time_to_first_hint_hours`는 해당 이벤트가 첫 24시간에 발생하지 않은 사용자에게 `NULL`로 둔다. 0은 “T0 직후 발생”이라는 실제 의미가 있으므로 미발생 사용자를 0으로 채우면 안 된다.

모델링 단계에서는 시간 피처의 결측값을 별도로 대치하되, 발생 여부 플래그와 함께 사용한다.

## 6. 힌트 피처 테이블 생성

모든 행동 조건은 `feature_start_at` 이상, `feature_end_at` 미만으로 제한한다. D+7 타깃 구간의 정보는 사용하지 않는다.

In [ ]:
query = """
CREATE OR REPLACE TABLE `your-gcp-project.mart.fct_ml_hint_24h` AS

WITH activation_ts AS (
    SELECT
        user_id,
        TIMESTAMP(first_receive_at, 'Asia/Seoul') AS first_receive_at_ts,
        TIMESTAMP(first_hint_opened_at, 'Asia/Seoul') AS first_hint_opened_at_ts
    FROM `your-gcp-project.mart.mart_activation`
)

SELECT
    b.user_id,

    -- QA용: 최종 모델 피처에서는 제외
    IF(a.user_id IS NOT NULL, 1, 0) AS activation_match_flag,

    -- T0 이전에 이미 서비스 최초 수신이 발생했는지
    IF(a.first_receive_at_ts < b.feature_start_at, 1, 0)
        AS had_received_before_t0,

    -- 서비스 최초 수신이 T0 이후 24시간 안에 발생했는지
    IF(
        a.first_receive_at_ts >= b.feature_start_at
        AND a.first_receive_at_ts < b.feature_end_at,
        1,
        0
    ) AS first_receive_in_24h,

    CASE
        WHEN a.first_receive_at_ts >= b.feature_start_at
         AND a.first_receive_at_ts < b.feature_end_at
        THEN TIMESTAMP_DIFF(
            a.first_receive_at_ts,
            b.feature_start_at,
            MINUTE
        ) / 60.0
    END AS time_to_first_receive_hours,

    -- T0 이전에 이미 서비스 최초 힌트 열람이 발생했는지
    IF(a.first_hint_opened_at_ts < b.feature_start_at, 1, 0)
        AS had_hint_before_t0,

    -- 서비스 최초 힌트 열람이 T0 이후 24시간 안에 발생했는지
    IF(
        a.first_hint_opened_at_ts >= b.feature_start_at
        AND a.first_hint_opened_at_ts < b.feature_end_at,
        1,
        0
    ) AS first_hint_opened_in_24h,

    CASE
        WHEN a.first_hint_opened_at_ts >= b.feature_start_at
         AND a.first_hint_opened_at_ts < b.feature_end_at
        THEN TIMESTAMP_DIFF(
            a.first_hint_opened_at_ts,
            b.feature_start_at,
            MINUTE
        ) / 60.0
    END AS time_to_first_hint_hours,

    -- 24시간 내 최초 힌트 열람보다 먼저 PING를 받은 경험이 있는지
    IF(
        a.first_hint_opened_at_ts >= b.feature_start_at
        AND a.first_hint_opened_at_ts < b.feature_end_at
        AND a.first_receive_at_ts IS NOT NULL
        AND a.first_receive_at_ts <= a.first_hint_opened_at_ts,
        1,
        0
    ) AS received_before_first_hint

FROM `your-gcp-project.mart.mart_ml_d7_base` b
LEFT JOIN activation_ts a USING (user_id)
"""

client.query(query).result()
print("힌트 피처 테이블 생성 완료")

## 7. 생성 결과 QA

다음 조건을 모두 만족해야 한다.

1. `row_count = user_count = 4,844`
2. `duplicate_count = 0`
3. 24시간 내 이벤트 플래그가 1이면 해당 시간 피처는 `NULL`이 아니다.
4. 24시간 내 이벤트 플래그가 0이면 해당 시간 피처는 `NULL`이다.
5. 시간값은 0 이상 24 미만이다.

In [ ]:
query = """
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT user_id) AS user_count,
    COUNT(*) - COUNT(DISTINCT user_id) AS duplicate_count,
    COUNTIF(activation_match_flag = 0) AS no_activation_match,

    COUNTIF(first_receive_in_24h = 1) AS receive_user_count,
    COUNTIF(first_hint_opened_in_24h = 1) AS hint_user_count,

    COUNTIF(
        first_receive_in_24h = 1
        AND time_to_first_receive_hours IS NULL
    ) AS receive_flag_time_error,
    COUNTIF(
        first_receive_in_24h = 0
        AND time_to_first_receive_hours IS NOT NULL
    ) AS receive_no_flag_time_error,

    COUNTIF(
        first_hint_opened_in_24h = 1
        AND time_to_first_hint_hours IS NULL
    ) AS hint_flag_time_error,
    COUNTIF(
        first_hint_opened_in_24h = 0
        AND time_to_first_hint_hours IS NOT NULL
    ) AS hint_no_flag_time_error,

    COUNTIF(
        time_to_first_receive_hours < 0
        OR time_to_first_receive_hours >= 24
    ) AS receive_time_range_error,
    COUNTIF(
        time_to_first_hint_hours < 0
        OR time_to_first_hint_hours >= 24
    ) AS hint_time_range_error

FROM `your-gcp-project.mart.fct_ml_hint_24h`
"""

feature_qa = client.query(query).to_dataframe()
feature_qa.T

In [ ]:
query = """
SELECT *
FROM `your-gcp-project.mart.fct_ml_hint_24h`
ORDER BY user_id
LIMIT 10
"""

hint_feature_sample = client.query(query).to_dataframe()
hint_feature_sample

## 8. 최종 해석 및 전달 사항

이 테이블은 Base Mart의 4,844명을 유지한 사용자 단위 힌트 피처 테이블이다. 최종 Training Mart에서는 다음과 같이 `LEFT JOIN`한다.

```sql
LEFT JOIN `your-gcp-project.mart.fct_ml_hint_24h` h USING (user_id)
```

### 모델링 시 주의사항

- `activation_match_flag`는 데이터 매칭 상태를 확인하기 위한 QA 컬럼이므로 모델 입력에서 제외한다.
- 시간 피처의 `NULL`을 0으로 채우지 않는다. 전처리 Pipeline에서 결측값을 대치하고 대응하는 이벤트 발생 플래그를 함께 사용한다.
- `first_hint_opened_in_24h`는 “24시간 내 힌트를 한 번이라도 열었는가”가 아니라 “서비스 최초 힌트 열람이 24시간 내 발생했는가”를 의미한다.
- 향후 힌트 열람 건별 이벤트 시각이 확보되면 `hint_open_count_24h`, 실제 `used_hint_24h`를 추가할 수 있다.

### 한계

`mart_activation`의 최초 시각만 사용하므로 T0 이전에 이미 힌트 또는 수신 경험이 있는 사용자의 **T0 이후 추가 행동**은 측정할 수 없다. 따라서 이 피처는 초기 경험의 순서와 시점을 나타내는 제한적인 대리변수로 해석해야 한다.